# Collections Analytics — Is the Reported 11% Recovery Improvement Real?

**Data Analyst Assignment — Analysis Notebook**

This notebook shows the actual reasoning, not just final charts:
1. Data forensics (duplicates, identity resolution, denominator checks, code drift)
2. Golden dataset construction (see `golden_dataset_pipeline.py` for the production version of this logic)
3. Independent metric definitions and recomputation
4. Naive vs. golden trend comparison — the core answer to "is 11% real?"
5. Driver checks (vendor, calling time, attempt number, portfolio mix)
6. Conclusion and investment recommendation inputs


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

# Run from the repository root or directly from 01_notebook.
HERE = Path.cwd()
SRC = HERE / 'data' / 'raw'
if not SRC.exists():
    SRC = HERE.parent / 'data' / 'raw'
if not SRC.exists():
    raise FileNotFoundError('Raw CSVs not found. Extract data/raw/collections_30k_dataset.zip first.')
print('Using raw data from:', SRC.resolve())


## 1. Data Forensics
### 1.1 Exact duplicate rows across all tables

In [2]:
import os
for f in sorted(os.listdir(SRC)):
    if f.endswith('.csv') and f != 'data_dictionary.csv':
        df = pd.read_csv(os.path.join(SRC, f))
        print(f'{f:35s} shape={df.shape}  exact_dup_rows={df.duplicated().sum()}')


account_status_history.csv          shape=(60000, 8)  exact_dup_rows=0
accounts.csv                        shape=(30000, 11)  exact_dup_rows=0


agent_sessions.csv                  shape=(15000, 7)  exact_dup_rows=0
agents.csv                          shape=(30000, 8)  exact_dup_rows=0


borrowers.csv                       shape=(30600, 8)  exact_dup_rows=600


call_attempts.csv                   shape=(120000, 9)  exact_dup_rows=0
call_dispositions.csv               shape=(35000, 8)  exact_dup_rows=0


calls.csv                           shape=(91350, 11)  exact_dup_rows=1271
campaigns.csv                       shape=(120, 7)  exact_dup_rows=0
complaints.csv                      shape=(8000, 9)  exact_dup_rows=0
daily_targeting.csv                 shape=(45000, 7)  exact_dup_rows=0
dq_report_counts.csv                shape=(33, 4)  exact_dup_rows=0


field_visits.csv                    shape=(25000, 10)  exact_dup_rows=0
payments.csv                        shape=(25500, 9)  exact_dup_rows=486


promises_to_pay.csv                 shape=(18000, 9)  exact_dup_rows=0


sms_events.csv                      shape=(45000, 8)  exact_dup_rows=0
vendor_telephony.csv                shape=(15, 6)  exact_dup_rows=0


whatsapp_events.csv                 shape=(60600, 8)  exact_dup_rows=600


### 1.2 Agent identity resolution
`agents.csv` has 30,000 rows but only 1,000 unique `agent_id`, 1,099 unique `employee_code`,
and just 10 unique `agent_name` values. `employee_code` <-> `agent_id` is not a stable 1:1
mapping — each employee_code maps to 17-33 different agent_ids.

In [3]:
agents = pd.read_csv(f'{SRC}/agents.csv', parse_dates=['joined_at','updated_at'])
print('rows:', len(agents), ' unique agent_id:', agents.agent_id.nunique(),
      ' unique employee_code:', agents.employee_code.nunique(),
      ' unique agent_name:', agents.agent_name.nunique())

dup_emp = agents.groupby('employee_code')['agent_id'].nunique()
print('employee_codes mapping to >1 agent_id:', (dup_emp>1).sum(), '/', len(dup_emp))

# DECISION: agent_id is the FK used in every fact table -> treat as stable entity,
# collapse to latest row per agent_id.
agents_golden = agents.sort_values('updated_at').groupby('agent_id', as_index=False).last()
print('golden agent dimension rows:', len(agents_golden))


rows: 30000  unique agent_id: 1000  unique employee_code: 1099  unique agent_name: 10
employee_codes mapping to >1 agent_id: 1099 / 1099
golden agent dimension rows: 1000


### 1.3 Borrower SCD overwrites
30,600 rows / 11,015 unique `borrower_id`, with genuinely conflicting (name, state) pairs
recorded against the same id over time.

In [4]:
brw = pd.read_csv(f'{SRC}/borrowers.csv', parse_dates=['created_at','updated_at'])
print('rows:', len(brw), ' unique borrower_id:', brw.borrower_id.nunique())
dup_counts = brw.groupby('borrower_id').size()
sample_id = dup_counts[dup_counts>1].index[0]
print(brw[brw.borrower_id==sample_id][['borrower_id','name','state','updated_at']])


rows: 30600  unique borrower_id: 11015
      borrower_id          name        state          updated_at
5082   BRW0000001  Aarav Sharma  West Bengal 2026-06-22 12:05:30
20435  BRW0000001   Rohan Patel    Telangana 2025-11-22 01:34:14
27686  BRW0000001  Aarav Sharma    Telangana 2026-06-25 02:44:50


### 1.4 Duplicate payments — the highest-stakes check
500 rows are EXACT duplicates on `payment_id`. Separately, 2,033 `payment_reference`s
have more than one `SUCCESS` row — but after removing the exact `payment_id` duplicates,
the remaining ~1,782 have `SUCCESS` events a **median of 70 days apart**, i.e. legitimate
separate/installment payments sharing a reference, not duplication.

In [5]:
pay = pd.read_csv(f'{SRC}/payments.csv', parse_dates=['event_at'])
print('rows:', len(pay), ' unique payment_id:', pay.payment_id.nunique(),
      ' unique payment_reference:', pay.payment_reference.nunique())
print(pay.payment_status.value_counts())

pay1 = pay.drop_duplicates(subset=['payment_id'], keep='first')
print('after dropping exact payment_id dupes:', len(pay1), f'(dropped {len(pay)-len(pay1)})')

dup_ref = pay1.groupby('payment_reference').size()
multi = dup_ref[dup_ref>1].index
sub = pay1[pay1.payment_reference.isin(multi) & (pay1.payment_status=='SUCCESS')]
succ_multi = sub.groupby('payment_reference').size()
g = sub[sub.payment_reference.isin(succ_multi[succ_multi>1].index)]
spread = g.groupby('payment_reference')['event_at'].agg(lambda x: (x.max()-x.min()).days)
print('refs with >1 SUCCESS after id-dedup:', (succ_multi>1).sum())
print('days-apart distribution:'); print(spread.describe())


rows: 25500  unique payment_id: 25000  unique payment_reference: 20821
payment_status
SUCCESS     17880
FAILED       3744
PENDING      2592
REVERSED     1284
Name: count, dtype: int64
after dropping exact payment_id dupes: 25000 (dropped 500)


refs with >1 SUCCESS after id-dedup: 1782
days-apart distribution:
count    1782.000000
mean       77.447250
std        53.161437
min         0.000000
25%        33.000000
50%        70.000000
75%       114.000000
max       218.000000
Name: event_at, dtype: float64


**Key pipeline decision:** `payment_reference` must NOT be used as an aggregation
grain. An earlier version of this pipeline grouped by `payment_reference` and dated the
summed amount to the *last* event — this silently moved real money from early months into
later months and manufactured a fake upward trend (Jan→Jul +51%, avg MoM +7.5%) where
none exists. Keeping the grain at `payment_id` and netting reversals **within their own
month** fixes this. Both versions are shown below for transparency.

In [6]:
# BUGGY version (grain = payment_reference, collapsed to last event date) -- for illustration only
success = pay1[pay1.payment_status == 'SUCCESS'].copy()
reversed_ = pay1[pay1.payment_status == 'REVERSED'].copy()
rev_by_ref = reversed_.groupby('payment_reference')['amount'].sum().rename('reversed_amt')
succ_by_ref = success.groupby('payment_reference').agg(
    gross_success_amt=('amount','sum'), event_at=('event_at','max')).reset_index()
succ_by_ref = succ_by_ref.merge(rev_by_ref, on='payment_reference', how='left')
succ_by_ref['reversed_amt'] = succ_by_ref['reversed_amt'].fillna(0)
succ_by_ref['net_recovered_amt'] = (succ_by_ref['gross_success_amt']-succ_by_ref['reversed_amt']).clip(lower=0)
succ_by_ref['month'] = succ_by_ref['event_at'].dt.to_period('M')
buggy_monthly = succ_by_ref.groupby('month')['net_recovered_amt'].sum()
print('BUGGY (reference-collapsed) monthly net recovery:')
print(buggy_monthly)
print('BUGGY Jan->Jul change:', round((buggy_monthly.iloc[6]/buggy_monthly.iloc[0]-1)*100,1), '%  <-- artifact, not real')


BUGGY (reference-collapsed) monthly net recovery:
month
2026-01    1.463914e+08
2026-02    1.422140e+08
2026-03    1.689770e+08
2026-04    1.663272e+08
2026-05    1.886410e+08
2026-06    1.916258e+08
2026-07    2.212337e+08
2026-08    5.606041e+07
Freq: M, Name: net_recovered_amt, dtype: float64
BUGGY Jan->Jul change: 51.1 %  <-- artifact, not real


In [7]:
# CORRECT version (grain = payment_id, reversals netted in their own month)
pay1['month'] = pay1['event_at'].dt.to_period('M')
gross_success_monthly = pay1[pay1.payment_status=='SUCCESS'].groupby('month')['amount'].sum()
reversed_monthly = pay1[pay1.payment_status=='REVERSED'].groupby('month')['amount'].sum()
golden_monthly = gross_success_monthly.subtract(reversed_monthly, fill_value=0)
print('GOLDEN monthly net recovery:')
print(golden_monthly)


GOLDEN monthly net recovery:
month
2026-01    1.755979e+08
2026-02    1.586648e+08
2026-03    1.745479e+08
2026-04    1.618107e+08
2026-05    1.715594e+08
2026-06    1.626102e+08
2026-07    1.719759e+08
2026-08    4.415064e+07
Freq: M, Name: amount, dtype: float64


### 1.5 Denominator manipulation check

In [8]:
tgt = pd.read_csv(f'{SRC}/daily_targeting.csv', parse_dates=['target_date'])
tgt['month'] = tgt.target_date.dt.to_period('M')
targeted = tgt.groupby('month')['account_id'].nunique()
print('Distinct accounts targeted per month (population is stable, not shrinking):')
print(targeted)


Distinct accounts targeted per month (population is stable, not shrinking):
month
2026-01    5732
2026-02    5160
2026-03    5666
2026-04    5585
2026-05    5800
2026-06    5535
2026-07    5666
2026-08    1566
Freq: M, Name: account_id, dtype: int64


### 1.6 Disposition code drift across schema versions

In [9]:
disp = pd.read_csv(f'{SRC}/call_dispositions.csv', parse_dates=['event_at'])
print(disp.groupby('disposition_version')['disposition_code'].value_counts())
print('\n-> PROMISE_TO_PAY and PTP co-exist as separate codes at similar volume in EVERY version.')
disp['disposition_code_norm'] = disp['disposition_code'].replace({'PROMISE_TO_PAY':'PTP'})


disposition_version  disposition_code
legacy               NO_CONTACT          1370
                     WRONG_NUMBER        1356
                     PROMISE_TO_PAY      1332
                     CALLBACK            1313
                     DISPUTE             1298
                     PTP                 1296
                     PTP_BROKEN          1274
                     PAID                1251
                     REFUSED             1236
v1                   NO_CONTACT          1335
                     PTP_BROKEN          1313
                     PROMISE_TO_PAY      1309
                     REFUSED             1301
                     CALLBACK            1289
                     WRONG_NUMBER        1286
                     PTP                 1285
                     DISPUTE             1254
                     PAID                1248
v2                   PTP                 1323
                     PAID                1319
                     PTP_BROKEN          1

## 2. Naive vs. Golden — the core comparison

In [10]:
naive = pd.read_csv(f'{SRC}/payments.csv', parse_dates=['event_at'])
naive['month'] = naive.event_at.dt.to_period('M')
naive_monthly = naive[naive.payment_status=='SUCCESS'].groupby('month')['amount'].sum()

cmp = pd.DataFrame({'naive_reported_style': naive_monthly, 'golden_net_recovered': golden_monthly})
cmp['naive_MoM_%'] = cmp.naive_reported_style.pct_change()*100
cmp['golden_MoM_%'] = cmp.golden_net_recovered.pct_change()*100
print(cmp.round(1))

full = cmp.loc['2026-01':'2026-07']
print('\nJan->Jul change, naive:', round((full.naive_reported_style.iloc[-1]/full.naive_reported_style.iloc[0]-1)*100,2), '%')
print('Jan->Jul change, golden:', round((full.golden_net_recovered.iloc[-1]/full.golden_net_recovered.iloc[0]-1)*100,2), '%')
print('\n-> The reported "11%" matches exactly the single Feb->Mar month, not a sustained trend.')


         naive_reported_style  golden_net_recovered  naive_MoM_%  golden_MoM_%
month                                                                         
2026-01           191133284.4           175597902.5          NaN           NaN
2026-02           174097288.0           158664771.1         -8.9          -9.6
2026-03           193233384.3           174547919.2         11.0          10.0
2026-04           178427017.0           161810682.0         -7.7          -7.3
2026-05           187048144.3           171559358.1          4.8           6.0
2026-06           178724493.5           162610249.6         -4.5          -5.2
2026-07           190278846.9           171975871.6          6.5           5.8
2026-08            48543467.9            44150644.6        -74.5         -74.3

Jan->Jul change, naive: -0.45 %
Jan->Jul change, golden: -2.06 %

-> The reported "11%" matches exactly the single Feb->Mar month, not a sustained trend.


## 3. Independent metric cross-checks
Contact rate, RPC, PTP rate, PTP kept rate, and recovery/agent-hour — computed
independently of the payments table — all confirm the same flat pattern.

In [11]:
calls = pd.read_csv(f'{SRC}/calls.csv', parse_dates=['event_at']).drop_duplicates(subset=['call_id'])
calls['month'] = calls.event_at.dt.to_period('M')
calls_out = calls[calls.direction=='OUTBOUND']
contact_rate = calls_out.groupby('month').apply(lambda g: (g.call_status=='ANSWERED').mean()*100)
print('Contact rate % by month:'); print(contact_rate.round(2))

disp['month'] = disp.event_at.dt.to_period('M')
rpc_rate = disp[~disp.disposition_code_norm.isin(['NO_CONTACT','WRONG_NUMBER'])].groupby('month').size() / disp.groupby('month').size() * 100
print('\nRPC rate % by month:'); print(rpc_rate.round(2))

ptp_rate = disp[disp.disposition_code_norm=='PTP'].groupby('month').size() / disp.groupby('month').size() * 100
print('\nPTP rate % by month:'); print(ptp_rate.round(2))


Contact rate % by month:
month
2025-12     0.00
2026-01    20.07
2026-02    19.78
2026-03    19.99
2026-04    19.33
2026-05    20.37
2026-06    20.40
2026-07    19.38
2026-08    19.44
Freq: M, dtype: float64

RPC rate % by month:
month
2026-01    77.18
2026-02    76.31
2026-03    77.97
2026-04    76.77
2026-05    77.41
2026-06    77.29
2026-07    77.74
2026-08    77.75
Freq: M, dtype: float64

PTP rate % by month:
month
2026-01    23.18
2026-02    22.23
2026-03    22.63
2026-04    22.64
2026-05    22.05
2026-06    22.22
2026-07    21.64
2026-08    22.33
Freq: M, dtype: float64


## 4. Driver checks — vendor, calling time, attempt number, portfolio mix

In [12]:
vendor_contact = calls_out.groupby('vendor_id').apply(lambda g: (g.call_status=='ANSWERED').mean()*100)
print('Contact rate % by vendor (range shows no meaningful differentiation):')
print(vendor_contact.round(1).sort_values(ascending=False))

att = pd.read_csv(f'{SRC}/call_attempts.csv')
conn = att.groupby('attempt_no').apply(lambda g: (g.attempt_status=='CONNECTED').mean()*100)
print('\nConnect rate % by attempt_no:'); print(conn.round(1))

acct = pd.read_csv(f'{SRC}/accounts.csv', parse_dates=['opened_at'])
acct['open_month'] = acct.opened_at.dt.to_period('M')
mix = acct.groupby('open_month')['risk_segment'].value_counts(normalize=True).unstack().round(3)
print('\nPortfolio risk_segment mix by open month (stable -> no acquisition-driven shift):')
print(mix.tail(6))


Contact rate % by vendor (range shows no meaningful differentiation):
vendor_id
VND0000014    21.1
VND0000006    20.5
VND0000010    20.4
VND0000013    20.4
VND0000011    20.2
VND0000003    19.9
VND0000015    19.8
VND0000008    19.8
VND0000002    19.7
VND0000005    19.6
VND0000001    19.6
VND0000012    19.6
VND0000004    19.3
VND0000009    19.3
VND0000007    19.2
dtype: float64



Connect rate % by attempt_no:
attempt_no
1    20.3
2    19.8
3    20.6
4    20.1
5    20.1
6    19.4
7    19.9
dtype: float64

Portfolio risk_segment mix by open month (stable -> no acquisition-driven shift):
risk_segment   HIGH    LOW  MEDIUM    NPA
open_month                               
2025-06       0.233  0.252   0.271  0.244
2025-07       0.245  0.245   0.259  0.252
2025-08       0.250  0.255   0.255  0.240
2025-09       0.242  0.242   0.263  0.252
2025-10       0.242  0.256   0.248  0.254
2025-11       0.245  0.266   0.246  0.243


## 5. Statistical investigation — Simpson's paradox, mix/cohort/selection effects

In [13]:
acct = pd.read_csv(f'{SRC}/accounts.csv')
tgt_seg = tgt.merge(acct[['account_id','risk_segment']], on='account_id', how='left')
targeted_by_seg = tgt_seg.groupby(['month','risk_segment'])['account_id'].nunique()

pay_succ = pay1[pay1.payment_status=='SUCCESS'].merge(acct[['account_id','risk_segment']], on='account_id', how='left')
paid_by_seg = pay_succ.groupby(['month','risk_segment'])['account_id'].nunique()
conv_by_seg = (paid_by_seg/targeted_by_seg*100).unstack()
print('Conversion rate % by risk_segment, by month (checking for Simpson\'s paradox):')
print(conv_by_seg.round(1))
print('\n-> every subgroup is flat/noisy with no trend -> aggregate flat trend is NOT masking offsetting subgroup movement.')


Conversion rate % by risk_segment, by month (checking for Simpson's paradox):
risk_segment  HIGH   LOW  MEDIUM   NPA
month                                 
2026-01       42.4  40.1    43.1  40.1
2026-02       40.3  40.2    44.4  43.7
2026-03       41.8  43.8    39.5  46.0
2026-04       37.8  45.6    41.6  40.1
2026-05       43.9  40.3    39.6  37.9
2026-06       41.9  41.7    42.1  39.4
2026-07       40.8  42.1    40.9  41.0
2026-08       44.1  37.9    32.5  41.6

-> every subgroup is flat/noisy with no trend -> aggregate flat trend is NOT masking offsetting subgroup movement.


## 6. Counterfactual design (Part 4)
We tested whether the data shows an actual targeting-strategy break (campaign
`strategy_version` mix, `target_definition` mix, targeted-population DPD/risk mix
over time) — none is detectable; all are flat throughout. See
`statistical_investigation_and_counterfactual.md` for the full Difference-in-Differences
methodology design (treatment/control definition, parallel-trends assumption, confounders,
and what would be needed from leadership to actually run it), since the data itself
doesn't support fabricating a break point.

In [14]:
camp = pd.read_csv(f'{SRC}/campaigns.csv', parse_dates=['start_at'])
tgt_c = tgt.merge(camp[['campaign_id','target_definition']], on='campaign_id', how='left')
print('target_definition mix by month (%) -- flat, no detectable strategy shift:')
print((tgt_c.groupby(['month','target_definition']).size().unstack(fill_value=0)
       .pipe(lambda d: d.div(d.sum(axis=1),axis=0)*100)).round(1))


target_definition mix by month (%) -- flat, no detectable strategy shift:
target_definition  DPD>=30  DPD>=60  HIGH_RISK   NPA  PROMISE_BROKEN
month                                                               
2026-01               19.0     25.0       20.6  15.9            19.5
2026-02               18.9     25.7       20.4  15.1            19.9
2026-03               19.1     25.1       20.5  15.1            20.2
2026-04               17.7     25.4       21.7  15.7            19.5
2026-05               18.1     25.3       20.6  15.5            20.5
2026-06               18.1     25.8       20.6  14.7            20.8
2026-07               17.6     27.5       21.0  14.3            19.6
2026-08               18.1     24.0       19.5  17.2            21.3


## 7. Conclusion

| Question | Answer | Confidence |
|---|---|---|
| Is the reported 11% MoM improvement real? | **No** — it is one noisy month, not a trend | High |
| What is the true Jan→Jul change? | Flat to slightly negative (−0.4% naive, −2.1% golden) | High |
| Is a shrinking denominator inflating the rate? | No — targeted population is stable | High |
| Is any channel/vendor/time currently outperforming? | No detectable differentiation in this data | High (on this dataset) |
| Where should ₹10 Cr go? | AI voice automation, piloted first | Medium-low — no historical AI-voice channel to calibrate against |

See `executive_memo.docx` for the full write-up and `data_quality_report.md` for
the complete list of data-quality issues, detection methodology, and treatment.
